<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_5/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_5_2_%D0%98%D0%BD%D1%82%D0%B5%D0%B3%D1%80%D0%B0%D1%86%D0%B8%D1%8F_RAG_%D1%81_LLM_%D0%B8_%D0%BE%D0%BF%D1%82%D0%B8%D0%BC%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 5.2 – Интеграция RAG с LLM и оптимизация

## Тема 1. Подключение LLM к RAG-системе (Расширенное практическое руководство)

В предыдущей лекции мы построили полноценный RAG-пайплайн: загрузили документы, разбили на чанки, сгенерировали эмбеддинги и организовали векторный поиск. Однако до сих пор мы не подключали языковую модель (LLM) для генерации связного ответа. В этой теме мы разберём, как выбрать подходящую LLM, как правильно формировать промпт с контекстом, как интегрировать локальные модели через Ollama, а также как работать с облачными API — включая **бесплатные альтернативы**. Мы создадим единую абстракцию для разных провайдеров и покажем полный пример работы RAG с LLM.

---

### 1.1. Выбор LLM для генерации ответов (Детальный обзор)

Выбор языковой модели – одно из ключевых решений при проектировании RAG-системы. Он влияет на качество ответов, стоимость, скорость и требования к инфраструктуре. Рассмотрим все категории моделей, включая **полностью бесплатные варианты**, которые доступны каждому разработчику.

#### 1.1.1. Локальные модели (Ollama, LM Studio, llama.cpp)

**Что это:** Модели, которые вы запускаете на своём оборудовании (собственный сервер, рабочая станция или облачный инстанс с GPU). Вы скачиваете веса и запускаете через оптимизированные фреймворки.

**Преимущества:**
- **Полная приватность** – данные не покидают вашу инфраструктуру (критично для медицины, финансов, госсектора).
- **Бесплатность** – нет платы за токены, только стоимость железа (одноразовая или арендная).
- **Независимость от интернета** – работает даже в изолированных сетях.
- **Гибкость** – можно тонко настраивать параметры, дообучать модели.

**Недостатки:**
- Требуют мощного GPU (для 7B+ нужна видеокарта с 8–24 ГБ VRAM).
- Качество может уступать топовым облачным моделям (GPT‑4o, Claude).
- Ручное обновление версий.

**Инструменты для локального запуска:**
- **Ollama** – простой установщик, единый API, поддержка GPU и CPU. Самый популярный выбор для прототипов.
- **LM Studio** – графический интерфейс, удобен для экспериментов.
- **llama.cpp** – высокопроизводительный бэкенд для CPU, поддерживает квантизацию (int4, int8).

#### 1.1.2. Облачные API (платные и бесплатные)

Облачные API делятся на **платные** (с оплатой за токены) и **бесплатные** (с ограничениями по частоте запросов или модели).

**Платные провайдеры (высокое качество, масштабируемость):**

| Провайдер | Модель | Контекстное окно | Цена (вход, 1M) | Цена (выход, 1M) | Особенности |
|-----------|--------|------------------|-----------------|------------------|-------------|
| OpenAI | GPT-4o | 128K | $5.00 | $15.00 | Лучшее качество, мультимодальность |
| OpenAI | GPT-4o-mini | 128K | $0.15 | $0.60 | Лучшее соотношение цена/качество |
| Anthropic | Claude 3.5 Sonnet | 200K | $3.00 | $15.00 | Отличное рассуждение, большой контекст |
| Anthropic | Claude 3 Haiku | 200K | $0.25 | $1.25 | Очень быстрый и дешёвый |
| Cohere | Command R+ | 128K | $2.50 | $10.00 | Хорош для RAG, мультиязычный |
| Google | Gemini 1.5 Pro | 2M | $2.50 | $10.00 | Огромное окно, мультимодальный |

**Бесплатные облачные API (с ограничениями, но без оплаты):**

| Провайдер | Модель | Контекстное окно | Ограничения | Особенности |
|-----------|--------|------------------|-------------|-------------|
| **Google AI Studio** | Gemini 1.5 Flash | 1M | 15 запросов/мин, 1500 запросов/день | Бесплатно для исследователей, API-ключ легко получить |
| **Google AI Studio** | Gemini 1.5 Pro | 2M | 2 запроса/мин, 50 запросов/день | Огромное окно, мультимодальность |
| **Groq** | Llama 3.1 70B, Mixtral 8x7B | 128K | 30 запросов/мин (бесплатный уровень) | Очень высокая скорость (токенов/с) |
| **Hugging Face Inference API** | Множество открытых моделей | Зависит от модели | 30 запросов/мин (бесплатный токен) | Поддержка тысяч моделей, есть русскоязычные |
| **Together AI** | Llama, Mistral, Qwen | До 128K | Ограниченный бесплатный уровень | Предоставляет API к открытым моделям |
| **Replicate** | Llama, Mistral, и др. | Зависит от модели | Ограниченный бесплатный уровень | Хорошая документация, легко начать |
| **DeepSeek API** | DeepSeek-V2, DeepSeek-Coder | 128K | Бесплатно (во время бета-теста) | Отличное качество для кода и логики |

**Важно:** большинство бесплатных API имеют ограничения на количество запросов в минуту и в день, но для прототипов и небольших проектов их вполне достаточно.

#### 1.1.3. Открытые модели (запускаемые локально)

Модели с открытыми весами — это "золотая середина" между локальным запуском и облачными API. Вы скачиваете веса и запускаете на своём железе.

**Популярные семейства:**

- **Qwen 2.5 (Alibaba)** – лучшая поддержка русского языка, контекст 128K, размеры от 0.5B до 72B.
- **Llama 3.1 (Meta)** – отличное качество для английского, контекст 128K, размеры 8B, 70B, 405B.
- **Mistral** – компактные модели (7B, 8x7B) с высоким качеством и скоростью.
- **Phi-3 (Microsoft)** – маленькие модели (3.8B) с качеством близким к Llama 8B.

**Сравнительная таблица открытых моделей:**

| Модель | Размер | Контекст | MMLU | VRAM (min) | Языки |
|--------|--------|----------|------|------------|-------|
| Qwen 2.5 | 3B | 128K | ~70 | ~6 GB | русский, английский, 100+ |
| Qwen 2.5 | 7B | 128K | ~75 | ~14 GB | русский, английский, 100+ |
| Qwen 2.5 | 14B | 128K | ~78 | ~28 GB | русский, английский, 100+ |
| Llama 3.1 | 8B | 128K | ~73 | ~16 GB | английский (русский хуже) |
| Mistral | 7B | 32K | ~72 | ~14 GB | английский, французский |
| Phi-3 | 3.8B | 128K | ~69 | ~8 GB | английский |

#### 1.1.4. Критерии выбора модели (подробно)

**Качество** – оценивается по бенчмаркам (MMLU, GSM8K, MT-Bench). Для RAG важна точность следования инструкциям и работа с контекстом.

**Скорость** – измеряется в токенах в секунду. Для чат-систем важна скорость > 20 токенов/с.

**Стоимость** – для облачных: $/1M токенов; для локальных: стоимость оборудования. При большом количестве запросов локальные модели могут быть дешевле.

**Размер контекстного окна** – чем больше, тем больше чанков можно передать. Минимум 32K, оптимально 128K.

**Поддержка языка** – для русского языка лучшие открытые модели: Qwen 2.5, GigaChat, YandexGPT.

**Приватность** – если данные чувствительны, только локальные модели.

**Доступность железа** – наличие GPU (VRAM) определяет, какие локальные модели можно запускать.

**Рекомендации по выбору:**

| Сценарий | Рекомендация | Обоснование |
|----------|--------------|-------------|
| **Прототип, малый бюджет, локально** | Ollama + Qwen 2.5 3B | Бесплатно, достаточно качественно, простой запуск |
| **Прототип, без GPU** | Google Gemini 1.5 Flash (бесплатно) | Бесплатно, большое окно, хорошее качество |
| **Высокое качество, есть бюджет** | OpenAI GPT-4o или Claude 3.5 Sonnet | Лучшее качество, высокая скорость, обновления |
| **Приватность, хорошее качество** | Локально Qwen 2.5 7B или Llama 3.1 8B | Полный контроль, отличное качество |
| **Мультиязычность** | Qwen 2.5 (любой размер) | Лучшая поддержка русского |
| **Длинные документы (>32K)** | Qwen 2.5 7B (128K) или Llama 3.1 8B (128K) | Широкое окно |
| **Ограниченное железо (8GB VRAM)** | Qwen 2.5 3B или Phi-3 3.8B | Компактные модели |
| **Бесплатно, хорошая скорость** | Groq (Llama 3.1 70B) – бесплатный уровень | Очень быстрая инференс |

---

### 1.2. Формирование промпта с контекстом (Детальное руководство)

Промпт — это инструкция, которую мы передаём LLM. В RAG он включает системную установку, контекст из найденных документов и сам вопрос. Правильное построение промпта критически важно: от него зависит, насколько модель точно будет использовать контекст и давать правильные ответы.

#### Структура промпта

Промпт состоит из трёх частей:

1. **Системная инструкция** – задаёт роль модели, правила использования контекста, запрет на выдумывание.
2. **Контекст** – найденные ретривером чанки, отформатированные с указанием источника.
3. **Вопрос пользователя** – то, на что нужно ответить.

**Пример шаблона для фактологического вопроса:**

```
<|system|>
Ты — профессиональный консультант. Отвечай на вопросы, используя ТОЛЬКО предоставленный контекст. Если в контексте нет информации, скажи: "Я не знаю". Никогда не выдумывай факты. Всегда указывай источники информации.

<|context|>
[ДОКУМЕНТ 1] Источник: Налоговый кодекс РФ, глава 25
Ставка налога на прибыль составляет 20%...
[ДОКУМЕНТ 2] Источник: Закон о НПД
Самозанятые платят 4% или 6%...

<|question|>
Какой налог платят самозанятые?
<|answer|>
```

**Для сравнительного вопроса:**

```
<|system|>
Ты — аналитик. Сравни информацию из разных документов и выдели ключевые различия.

<|context|>
[ДОКУМЕНТ 1] Источник: Ставки для ИП
...
[ДОКУМЕНТ 2] Источник: Ставки для самозанятых
...

<|question|>
Сравни налоговые ставки для ИП и самозанятых.
<|answer|>
```

**Для вопроса, требующего обобщения:**

```
<|system|>
Ты — эксперт. Обобщи информацию из нескольких источников и дай краткий вывод.

<|context|>
[ДОКУМЕНТ 1] ...
[ДОКУМЕНТ 2] ...
[ДОКУМЕНТ 3] ...

<|question|>
Какие основные изменения в налогообложении в 2025 году?
<|answer|>
```

#### Оформление документов в контексте

Каждый чанк должен содержать метаданные, чтобы модель могла сослаться на источник:

```
[ДОКУМЕНТ {номер}] Источник: {название} | Страница: {номер} | Дата: {дата}
{текст}
```

**Зачем это нужно:**
- Модель может указать источник в ответе → повышает доверие.
- Пользователь может проверить информацию.
- При ошибке можно идентифицировать проблемный документ.

#### Ограничения по длине контекста

Важно, чтобы общая длина промпта (система + контекст + вопрос + ответ) не превышала контекстное окно модели. Если превышает, модель обрежет часть контекста.

**Формула расчёта:**

```
total_tokens = len(system) + sum(len(chunk) for chunk in top_chunks) + len(question) + запас_на_ответ
```

- `len()` — число токенов (приблизительно количество символов / 4 для английского, / 3 для русского).
- Запас на ответ — минимум 200 токенов.

**Рекомендации:**
- Оставляйте минимум 20% окна для ответа.
- Если контекст слишком велик, уменьшите `top_k` (число возвращаемых чанков).
- Используйте чанки меньшего размера (200–300 токенов).
- Для моделей с малым окном (например, 4K) передавайте не более 2 чанков.

#### Способы сжатия контекста

1. **Динамическое усечение** – передаём только топ-K самых релевантных чанков.
2. **Суммаризация** – каждым чанк сжимается отдельной моделью (например, BART) до краткого абзаца.
3. **Выбор ключевых предложений** – оставляем только самые важные предложения из чанка.
4. **Иерархический подход** – сначала передаём заголовки/краткое содержание, затем по запросу детали.

---

### 1.3. Практическая интеграция с Ollama (Пошаговое руководство)

Ollama — самый простой способ запуска локальных LLM. Он предоставляет единый API для множества моделей и работает на всех основных ОС.

#### 1.3.1. Установка и настройка Ollama

**Установка на macOS и Linux:**

```bash
curl -fsSL https://ollama.com/install.sh | sh
```

**Установка на Windows:** скачайте установщик с официального сайта.

**Запуск сервера:**

```bash
ollama serve
```

**Загрузка моделей:**

```bash
# Для русского языка – Qwen 2.5 3B
ollama pull qwen2.5:3b

# Для английского – Llama 3.2 3B
ollama pull llama3.2:3b

# Для высокого качества (нужен GPU 16GB) – Qwen 2.5 7B
ollama pull qwen2.5:7b

# Просмотр установленных моделей
ollama list
```

**Проверка работы:**

```bash
ollama run qwen2.5:3b "Привет, как дела?"
```

#### 1.3.2. Отправка запроса через Python API

```python
import requests
import json
from typing import Optional

def query_ollama(prompt: str,
                 model: str = "qwen2.5:3b",
                 stream: bool = False,
                 temperature: float = 0.7,
                 max_tokens: int = 512,
                 system: Optional[str] = None) -> str:
    """
    Отправляет запрос к Ollama API.
    """
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens,
            "top_p": 0.9,
            "repeat_penalty": 1.1,
        }
    }
    if system:
        payload["system"] = system

    try:
        if stream:
            response = requests.post(url, json=payload, stream=True, timeout=60)
            response.raise_for_status()
            full_text = ""
            for line in response.iter_lines():
                if line:
                    data = json.loads(line)
                    if 'response' in data:
                        full_text += data['response']
                    if data.get('done', False):
                        break
            return full_text
        else:
            response = requests.post(url, json=payload, timeout=60)
            response.raise_for_status()
            data = response.json()
            return data.get('response', '')
    except requests.exceptions.ConnectionError:
        return "❌ Ошибка: не удалось подключиться к Ollama. Убедитесь, что сервер запущен."
    except Exception as e:
        return f"❌ Ошибка: {str(e)}"

# Пример использования
response = query_ollama("Назови столицу Франции", model="qwen2.5:3b")
print(response)
```

#### 1.3.3. Потоковая передача (stream=True)

```python
def query_ollama_stream(prompt: str,
                        model: str = "qwen2.5:3b",
                        temperature: float = 0.7,
                        max_tokens: int = 512,
                        system: Optional[str] = None):
    """
    Возвращает генератор для потоковой передачи токенов.
    """
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens,
            "top_p": 0.9,
            "repeat_penalty": 1.1,
        }
    }
    if system:
        payload["system"] = system

    with requests.post(url, json=payload, stream=True, timeout=60) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if line:
                data = json.loads(line)
                if 'response' in data:
                    yield data['response']
                if data.get('done', False):
                    break

# Использование
def generate_with_streaming(prompt: str, model: str = "qwen2.5:3b"):
    print("🤖 Генерация: ", end="", flush=True)
    for token in query_ollama_stream(prompt, model=model):
        print(token, end="", flush=True)
    print("\n")
```

#### 1.3.4. Парсинг ответа

```python
def parse_ollama_response(raw: str) -> dict:
    # Удаляем управляющие символы, лишние пробелы
    cleaned = ' '.join(raw.strip().split())
    try:
        if cleaned.startswith('{') and cleaned.endswith('}'):
            return json.loads(cleaned)
    except:
        pass
    return {"text": cleaned}
```

#### 1.3.5. Полный рабочий пример в Google Colab

Ниже представлен **полный код RAG-системы с локальной LLM через Ollama**, который:
- Автоматически устанавливает Ollama в Colab
- Загружает выбранную модель
- Создаёт векторную БД в Chroma
- Выполняет RAG-запросы с формированием контекста
- Работает без API-ключей
- Сохраняет БД на Google Drive
- Кэширует эмбеддинги для ускорения
- Имеет механизм повторных попыток при таймаутах

```python
# ================================================================
# RAG-система с локальными LLM через Ollama
# Исправленная версия с подавлением предупреждений и GPU
# ================================================================

!pip install -q requests sentence-transformers chromadb torch

import os
import sys
import time
import json
import requests
import subprocess
import warnings
import torch
from typing import Optional, List, Dict
import chromadb
from sentence_transformers import SentenceTransformer

# Подавление предупреждения UNEXPECTED (не влияет на работу)
warnings.filterwarnings("ignore", message=".*UNEXPECTED.*")

# ========== 1. ОПРЕДЕЛЕНИЕ СРЕДЫ ==========
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
print(f"🌐 Среда: {'Google Colab' if IS_COLAB else 'Локально'}")

# ========== 2. НАСТРОЙКА ХРАНИЛИЩА ==========
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DB_PATH = "/content/drive/MyDrive/chroma_db"
    print("✅ Google Drive смонтирован, БД будет сохранена в MyDrive")
else:
    DB_PATH = "./chroma_db"

# ========== 3. ВЫБОР МОДЕЛИ ==========
MODEL_NAME = "qwen2.5:1.5b"  # Можно заменить на "deepseek-r1:1.5b"
print(f"🤖 Используемая модель: {MODEL_NAME}")

# ========== 4. УСТАНОВКА И ЗАПУСК OLLAMA ==========
def ensure_model(model_name):
    """Проверяет, загружена ли модель, и скачивает при необходимости."""
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            models = [m['name'] for m in r.json().get('models', [])]
            if model_name in models:
                print(f"✅ Модель {model_name} уже загружена")
                return True
            else:
                print(f"📥 Модель {model_name} не найдена, скачиваем...")
                result = subprocess.run(["ollama", "pull", model_name], capture_output=True, text=True)
                if result.returncode == 0:
                    print(f"✅ Модель {model_name} успешно загружена")
                    return True
                else:
                    print(f"❌ Ошибка при загрузке: {result.stderr}")
                    return False
    except Exception as e:
        print(f"⚠️ Не удалось проверить модели: {e}")
        return False

def setup_ollama():
    if not IS_COLAB:
        try:
            subprocess.run(["ollama", "--version"], check=True, capture_output=True)
            print("✅ Ollama уже установлен локально")
            return ensure_model(MODEL_NAME)
        except:
            print("⚠️ Ollama не найден, установите вручную")
            return False

    # Проверяем, запущен ли сервер
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("✅ Ollama уже запущен")
        return ensure_model(MODEL_NAME)
    except:
        pass

    print("📦 Установка Ollama в Colab...")
    !apt-get install -y zstd pciutils > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh

    os.environ["PATH"] += os.pathsep + "/usr/local/bin"
    try:
        result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
        print(f"  ✅ Ollama установлен: {result.stdout.strip()}")
    except FileNotFoundError:
        print("  ❌ Ошибка установки")
        return False

    print("  → Запуск сервера...")
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True
    )
    print(f"  ✅ Сервер запущен (PID: {proc.pid})")

    for _ in range(30):
        try:
            requests.get("http://localhost:11434/api/tags", timeout=2)
            print("  ✅ Ollama готов!")
            break
        except:
            time.sleep(1)
    else:
        print("  ❌ Не дождались сервера")
        return False

    return ensure_model(MODEL_NAME)

OLLAMA_AVAILABLE = setup_ollama()
if not OLLAMA_AVAILABLE:
    print("❌ Не удалось запустить Ollama или загрузить модель.")
    sys.exit(1)

# ========== 5. ОБРАЩЕНИЕ К OLLAMA С ПОВТОРАМИ ==========
def query_ollama_with_retry(prompt, model=MODEL_NAME, temperature=0.7, max_tokens=512,
                            system=None, retries=3, timeout=300):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": max_tokens}
    }
    if system:
        payload["system"] = system

    for attempt in range(retries):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json().get("response", "")
        except requests.exceptions.Timeout:
            print(f"  ⏳ Таймаут (попытка {attempt+1}/{retries}), повтор через {2**attempt} сек...")
            time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  ❌ Ошибка: {e} (попытка {attempt+1}/{retries})")
            time.sleep(2 ** attempt)
    return "[Ошибка] Не удалось получить ответ после нескольких попыток."

class OllamaAdapter:
    def __init__(self, model=MODEL_NAME):
        self.model = model
    def generate(self, prompt, system=None, temperature=0.7, max_tokens=512):
        return query_ollama_with_retry(prompt, self.model, temperature, max_tokens, system)

# ========== 6. RAG СИСТЕМА ==========
class RAGSystem:
    def __init__(self, llm_adapter, embed_model_name="cointegrated/rubert-tiny2", db_path=DB_PATH):
        self.llm = llm_adapter
        # Используем GPU, если доступен
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🔧 Устройство для эмбеддингов: {device}")
        self.embed_model = SentenceTransformer(embed_model_name, device=device)
        self.db_path = db_path
        self.client = chromadb.PersistentClient(path=db_path)
        self.collection = None
        self._cache = {}
        self._init_collection()

    def _init_collection(self):
        try:
            self.collection = self.client.get_collection("documents")
            print(f"ℹ️ База 'documents' загружена, документов: {self.collection.count()}")
        except:
            self.collection = self.client.create_collection("documents")
            print("🆕 Создана новая коллекция 'documents'")

    def add_documents(self, documents: List[str], metadatas: List[Dict] = None):
        if metadatas is None:
            metadatas = [{} for _ in documents]

        embeddings = []
        for doc in documents:
            if doc not in self._cache:
                emb = self.embed_model.encode([doc], normalize_embeddings=True).tolist()[0]
                self._cache[doc] = emb
            embeddings.append(self._cache[doc])

        ids = [f"doc_{i}_{int(time.time())}" for i in range(len(documents))]
        self.collection.add(
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )
        print(f"✅ Добавлено {len(documents)} документов")

    def query(self, question: str, n_results: int = 3, system_prompt: str = None,
              max_context_tokens: int = 2000) -> str:
        q_emb = self.embed_model.encode([question], normalize_embeddings=True).tolist()
        results = self.collection.query(query_embeddings=q_emb, n_results=n_results)

        if not results['documents'] or not results['documents'][0]:
            return "❌ Нет релевантных документов."

        context = ""
        for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
            source = meta.get('source', 'неизвестно')
            context += f"\n[Документ {i+1}] Источник: {source}\n{doc}\n"

        # Усечение контекста, если он слишком длинный
        if len(context) // 4 > max_context_tokens:
            context = context[:max_context_tokens * 4] + "\n...[контекст обрезан]"

        default_system = ("Ты — юридический консультант. Отвечай строго по контексту. "
                          "Не добавляй свои знания. Если ответа нет в контексте, скажи: 'В контексте нет информации'.")
        system = system_prompt or default_system

        prompt = f"Контекст:\n{context}\n\nВопрос: {question}\nОтвет:"
        return self.llm.generate(prompt, system=system, temperature=0.3, max_tokens=512)

# ========== 7. ДЕМОНСТРАЦИЯ ==========
def demo():
    print("=" * 60)
    print("🧪 Проверка Ollama и модели")
    print("=" * 60)

    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        models = [m['name'] for m in r.json().get('models', [])]
        print(f"✅ Ollama доступен. Загруженные модели: {', '.join(models)}")
        if MODEL_NAME not in models:
            print(f"⚠️ Модель {MODEL_NAME} не найдена, попытка загрузить...")
            ensure_model(MODEL_NAME)
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return

    llm = OllamaAdapter(MODEL_NAME)
    rag = RAGSystem(llm)

    if rag.collection.count() == 0:
        print("🛠️ Наполняем базу тестовыми документами...")
        docs = [
            "Самозанятые граждане платят налог на профессиональный доход (НПД). Ставка 4% при работе с физлицами и 6% с юрлицами.",
            "Для российских IT-компаний с аккредитацией Минцифры налог на прибыль 0% до конца 2024, с 2025 — 5%.",
            "Страховые взносы для самозанятых необязательны, можно платить добровольно для пенсии.",
            "IT-компании освобождены от НДС при продаже собственного ПО."
        ]
        metas = [
            {"source": "Закон о НПД (ФЗ-422)"},
            {"source": "НК РФ (Ст. 284.5)"},
            {"source": "Закон о НПД (Ст. 14)"},
            {"source": "НК РФ (Ст. 145.1)"}
        ]
        rag.add_documents(docs, metas)

    questions = [
        "Какой налог платят самозанятые?",
        "Ставка налога на прибыль для IT-компаний?",
        "Обязательны ли страховые взносы для самозанятых?"
    ]

    print("\n" + "=" * 60)
    print("🔎 RAG-запросы")
    print("=" * 60)
    for q in questions:
        print(f"\n📌 Вопрос: {q}")
        answer = rag.query(q)
        print(f"   Ответ: {answer[:600]}{'...' if len(answer)>600 else ''}")

if __name__ == "__main__":
    demo()
```

**Результат выполнения:**

```
🌐 Среда: Google Colab
✅ Google Drive смонтирован, БД будет сохранена в MyDrive
🤖 Используемая модель: qwen2.5:1.5b
✅ Ollama уже запущен
✅ Модель qwen2.5:1.5b уже загружена
============================================================
🧪 Проверка Ollama и модели
============================================================
✅ Ollama доступен. Загруженные модели: qwen2.5:1.5b
🔧 Устройство для эмбеддингов: cuda
ℹ️ База 'documents' загружена, документов: 4
============================================================
🔎 RAG-запросы
============================================================

📌 Вопрос: Какой налог платят самозанятые?
   Ответ: Самозанятые граждане платят налог на профессиональный доход (НПД). Ставка 4% при работе с физлицами и 6% с юрлицами.

📌 Вопрос: Ставка налога на прибыль для IT-компаний?
   Ответ: Для российских IT-компаний с аккредитацией Минцифры налог на прибыль 0% до конца 2024, с 2025 — 5%.

📌 Вопрос: Обязательны ли страховые взносы для самозанятых?
   Ответ: Страховые взносы для самозанятых необязательны, можно платить добровольно для пенсии.
```

---

### 1.4. Интеграция с другими LLM API (включая бесплатные)

#### 1.4.1. Бесплатные облачные API

**Google Gemini API (бесплатный уровень)**

```python
import google.generativeai as genai

class GeminiAdapter:
    def __init__(self, api_key: str, model: str = "gemini-1.5-flash"):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model)

    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 512) -> str:
        response = self.model.generate_content(
            prompt,
            generation_config={"temperature": temperature, "max_output_tokens": max_tokens}
        )
        return response.text
```

**Groq API (бесплатный уровень, очень быстрый)**

```python
from groq import Groq

class GroqAdapter:
    def __init__(self, api_key: str, model: str = "llama3-70b-8192"):
        self.client = Groq(api_key=api_key)
        self.model = model

    def generate(self, prompt: str, system: str = None, temperature: float = 0.7, max_tokens: int = 512) -> str:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
```

**Hugging Face Inference API (бесплатный токен)**

```python
import requests

class HuggingFaceAdapter:
    def __init__(self, api_key: str, model: str = "meta-llama/Llama-3.2-3B-Instruct"):
        self.api_key = api_key
        self.model = model
        self.api_url = f"https://api-inference.huggingface.co/models/{model}"

    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 512) -> str:
        headers = {"Authorization": f"Bearer {self.api_key}"}
        payload = {
            "inputs": prompt,
            "parameters": {"temperature": temperature, "max_new_tokens": max_tokens}
        }
        response = requests.post(self.api_url, headers=headers, json=payload)
        return response.json()[0]['generated_text']
```

#### 1.4.2. Платные облачные API (OpenAI, Anthropic, Cohere)

**OpenAI API:**

```python
import openai

class OpenAIAdapter:
    def __init__(self, api_key: str, model: str = "gpt-4o-mini"):
        self.client = openai.OpenAI(api_key=api_key)
        self.model = model

    def generate(self, prompt: str, system: str = None, temperature: float = 0.7, max_tokens: int = 512) -> str:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
```

**Anthropic Claude API:**

```python
import anthropic

class ClaudeAdapter:
    def __init__(self, api_key: str, model: str = "claude-3-haiku-20240307"):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.model = model

    def generate(self, prompt: str, system: str = None, temperature: float = 0.7, max_tokens: int = 512) -> str:
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system or "You are a helpful assistant.",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
```

**Cohere API:**

```python
import cohere

class CohereAdapter:
    def __init__(self, api_key: str, model: str = "command-r"):
        self.client = cohere.Client(api_key)
        self.model = model

    def generate(self, prompt: str, system: str = None, temperature: float = 0.7, max_tokens: int = 512) -> str:
        response = self.client.chat(
            model=self.model,
            message=prompt,
            preamble=system or "You are a helpful assistant.",
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.text
```

#### 1.4.3. Единая абстракция (Factory Pattern)

```python
class LLMAdapter:
    @staticmethod
    def create(provider: str, api_key: str = None, model: str = None):
        if provider == "ollama":
            return OllamaAdapter(model or "qwen2.5:3b")
        elif provider == "openai":
            return OpenAIAdapter(api_key, model or "gpt-4o-mini")
        elif provider == "claude":
            return ClaudeAdapter(api_key, model or "claude-3-haiku-20240307")
        elif provider == "cohere":
            return CohereAdapter(api_key, model or "command-r")
        elif provider == "gemini":
            return GeminiAdapter(api_key, model or "gemini-1.5-flash")
        elif provider == "groq":
            return GroqAdapter(api_key, model or "llama3-70b-8192")
        elif provider == "huggingface":
            return HuggingFaceAdapter(api_key, model or "meta-llama/Llama-3.2-3B-Instruct")
        else:
            raise ValueError(f"Unknown provider: {provider}")
```

---

### 1.5. Пример: полный RAG-запрос с Ollama

Функция `rag_with_llm` демонстрирует полный цикл RAG-запроса: поиск → формирование контекста → генерация ответа.

```python
def rag_with_llm(question, collection, embed_model, llm_adapter, top_k=3):
    # 1. Поиск
    q_emb = embed_model.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=top_k)

    if not results['documents'] or len(results['documents'][0]) == 0:
        return {"answer": "Документы не найдены.", "sources": []}

    # 2. Контекст
    context = ""
    sources = []
    for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        src = meta.get('source', 'неизвестный')
        sources.append(src)
        context += f"\n[ДОКУМЕНТ {i+1}] Источник: {src}\n{doc}\n"

    # 3. Промпт
    system = "Ты — консультант. Используй только контекст. Ссылайся на источники."
    prompt = f"Контекст:\n{context}\n\nВопрос: {question}\nОтвет:"

    # 4. Генерация
    answer = llm_adapter.generate(prompt, system=system, temperature=0.3, max_tokens=512)
    return {"answer": answer, "sources": sources}
```

---

### 1.6. Контрольные вопросы

1. *Какие бесплатные облачные API можно использовать для RAG?*  
   **Ответ:** Google Gemini (1.5 Flash/Pro), Groq (Llama 3.1, Mixtral), Hugging Face Inference API (сотни моделей), Together AI, Replicate, DeepSeek API. Все они имеют бесплатный уровень с ограничениями по частоте запросов.

2. *Как уменьшить длину контекста, если он не влезает в окно модели?*  
   **Ответ:** Уменьшить `top_k`, обрезать чанки, использовать суммаризацию, динамическое усечение, или перейти на модель с большим окном (например, Gemini 1.5 Pro с 2M токенов).

3. *В чём главное преимущество локального запуска Ollama перед облачными API?*  
   **Ответ:** Полная приватность данных, отсутствие платы за токены, независимость от интернета. Недостаток – требуется мощное железо и ручное обновление.

---

### 1.7. Задания

1. **Функция query_llm для Ollama** – реализуйте и протестируйте на трёх разных типах промптов (фактический, творческий, инструкция).

2. **Адаптер для любого бесплатного API** – выберите любой бесплатный провайдер (Gemini, Groq, Hugging Face), реализуйте адаптер и сравните время ответа с Ollama на 5 одинаковых запросах. Запишите среднее время и сделайте вывод.

---

### 1.8. Список литературы

- Ollama: https://github.com/ollama/ollama
- Google Gemini API: https://ai.google.dev/
- Groq API: https://groq.com/
- Hugging Face Inference API: https://huggingface.co/docs/api-inference/index
- OpenAI API: https://platform.openai.com/docs
- Anthropic Claude: https://docs.anthropic.com/claude/reference

# Лекция 5.2 – Интеграция RAG с LLM и оптимизация

## Тема 2. Расширенный поиск и переранжирование

После того как мы настроили базовый RAG-пайплайн с векторным поиском и генерацией через LLM, возникает вопрос: **как улучшить качество поиска?** Чисто семантический поиск на эмбеддингах хорошо улавливает смысл, но часто пропускает точные совпадения (коды, номера, имена). Гибридный поиск, сочетающий семантику и лексику, закрывает эту брешь. Ещё один шаг — **переранжирование** (re-ranking) с помощью cross‑encoders, которое уточняет порядок найденных документов и может повысить метрики на 5–15%. В этой теме мы разберём все эти методы, их математику, реализацию на Python и сравним эффективность.

---

### 2.1. Гибридный поиск (семантический + ключевые слова)

#### Ограничения чистого семантического поиска

Векторный поиск на эмбеддингах отлично работает для запросов, сформулированных на естественном языке, где важны синонимы и контекст. Однако у него есть слабые места:

- **Пропускает точные совпадения** – если в запросе есть код продукта, номер документа или редкое слово, эмбеддинг может не отличить его от похожих семантических понятий.
- **Теряет редкие термины** – модели эмбеддингов обучаются на больших корпусах и могут недооценивать редко встречающиеся слова.
- **Не учитывает частоту терминов** – важность слова в документе не всегда коррелирует с его вкладом в эмбеддинг.

#### Добавление BM25

**BM25** (Best Matching 25) – это алгоритм ранжирования по релевантности, широко используемый в информационном поиске. Он оценивает, насколько хорошо документ соответствует запросу на основе частоты терминов.

**Формула BM25:**

Для запроса $Q$ с терминами $q_1, \dots, q_n$ и документа $D$:

$$
\text{BM25}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}
$$

где:

- $f(q_i, D)$ – частота термина $q_i$ в документе $D$,
- $|D|$ – длина документа (в словах),
- $\text{avgdl}$ – средняя длина документов в корпусе,
- $k_1$ и $b$ – параметры (обычно $k_1 = 1.2$, $b = 0.75$),
- $\text{IDF}(q_i)$ – обратная документная частота, обычно вычисляется как $\log\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5}\right)$,
  где $N$ – общее число документов, $n(q_i)$ – число документов, содержащих $q_i$.

BM25 не требует обучения и легко реализуется. Для Python есть библиотека `rank_bm25`.

#### Комбинирование результатов

Гибридный поиск объединяет оценки от семантического (векторного) и лексического (BM25) поиска. Один из простых способов – **взвешенная сумма**:

$$
\text{score}(D) = \alpha \cdot \text{score}_{\text{vector}}(D) + (1 - \alpha) \cdot \text{score}_{\text{BM25}}(D)
$$

где $\alpha$ — вес семантической составляющей (обычно 0.5–0.7). Оценки нужно нормализовать к одному диапазону (например, $[0, 1]$).

#### Реализация гибридного поиска

В нашем коде гибридный поиск реализован в классе `AdvancedRetriever`. Основные компоненты:

1. **Векторный поиск** через Chroma с косинусным сходством.
2. **BM25 поиск** через `rank_bm25.BM25Okapi`.
3. **Объединение** с взвешенной суммой.

```python
class AdvancedRetriever:
    def __init__(self, collection, embed_model, texts, metadatas,
                 alpha=0.6, k1=1.2, b=0.75,
                 rerank_top_k=20, final_top_k=5):
        self.collection = collection
        self.embed_model = embed_model
        self.texts = texts
        self.metadatas = metadatas
        self.alpha = alpha
        self.rerank_top_k = rerank_top_k
        self.final_top_k = final_top_k

        # Инициализация BM25 с параметрами
        tokenized = [doc.split() for doc in texts]
        self.bm25 = BM25Okapi(tokenized, k1=k1, b=b)
```

**Объединение оценок:**

```python
def search(self, query, filter_metadata=None):
    # Векторный поиск
    vec_ids, vec_scores = self._vector_search(query, self.rerank_top_k * 2)

    # BM25 поиск
    bm25_indices, all_scores = self._bm25_search(query, self.rerank_top_k * 2)
    max_bm25 = max(all_scores) if all_scores.any() else 1.0

    # Взвешенная сумма
    combined = {}
    for idx, score in zip(vec_ids, vec_scores):
        doc_idx = int(idx.split('_')[1])
        combined[doc_idx] = self.alpha * score

    for doc_idx in bm25_indices:
        score = all_scores[doc_idx] / max_bm25 if max_bm25 > 0 else 0
        combined[doc_idx] = combined.get(doc_idx, 0) + (1 - self.alpha) * score

    # Сортировка
    sorted_items = sorted(combined.items(), key=lambda x: x[1], reverse=True)
```

#### Когда гибридный поиск даёт лучшие результаты

- Запросы с **кодами, номерами** (например, "статья 284.5 НК РФ") – BM25 находит точное совпадение.
- Запросы с **именами собственными** (например, "ООО Ромашка").
- Запросы с **редкими терминами** – BM25 даёт им высокий вес.

---

### 2.2. Переранжирование результатов (Re-ranking)

#### Что такое реранкинг и зачем он нужен

После того как мы получили начальный набор документов (например, топ‑50 от гибридного поиска), мы можем применить более точную, но более медленную модель, чтобы переупорядочить их и выдать финальный топ‑5. Это называется **реранкинг**.

#### Cross‑encoders для точного ранжирования

В отличие от биэнкодеров (как Sentence‑BERT), которые кодируют запрос и документ отдельно, **cross‑encoder** принимает пару (запрос, документ) и выдаёт оценку релевантности. Это позволяет модели учитывать взаимодействие между словами запроса и документа, что даёт более точную оценку, но требует попарного вычисления.

**Популярные модели:**

- `cross-encoder/ms-marco-MiniLM-L-6-v2` – быстрая, обучена на данных поиска.
- `BAAI/bge-reranker-base` – более качественная, мультиязычная.

**Реализация реранкинга в коде:**

```python
# Загрузка cross-encoder
self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Применение к кандидатам
candidate_texts = [self.texts[idx] for idx, _ in candidates]
pairs = [[query, doc] for doc in candidate_texts]
rerank_scores = self.reranker.predict(pairs)

# Сортировка по новым оценкам
final_indices = np.argsort(rerank_scores)[::-1][:self.final_top_k]
```

#### Улучшение качества

Реранкинг может повысить метрики (например, Recall@5, MRR) на **5–15%** по сравнению с обычным гибридным поиском, особенно для сложных запросов, где начальное ранжирование не идеально.

#### Компромисс: реранкинг только для топ‑N

Так как cross‑encoder работает медленнее (попарно), его применяют только к небольшому числу кандидатов (например, топ‑50), чтобы не увеличивать время ответа.

```python
def __init__(self, ..., rerank_top_k=20, final_top_k=5):
    self.rerank_top_k = rerank_top_k
    self.final_top_k = final_top_k
```

---

### 2.3. Многопроходный поиск (Multi-stage retrieval)

Многопроходный поиск – это частный случай, когда мы используем несколько этапов для уточнения результатов.

**Структура в нашем коде:**

1. **Первый проход** – векторный поиск в Chroma (топ‑40).
2. **Второй проход** – BM25 (топ‑40).
3. **Третий проход** – объединение оценок и фильтрация.
4. **Четвёртый проход** – реранкинг cross‑encoder (топ‑20 → топ‑5).

```python
def search(self, query, filter_metadata=None):
    # 1. Векторный поиск (top_k * 2)
    vec_ids, vec_scores = self._vector_search(query, self.rerank_top_k * 2)

    # 2. BM25 поиск (top_k * 2)
    bm25_indices, all_scores = self._bm25_search(query, self.rerank_top_k * 2)

    # 3. Объединение и фильтрация
    # ... (взвешенная сумма, дедупликация, фильтрация)

    # 4. Реранкинг (топ-k -> final_top_k)
    candidates = sorted_items[:self.rerank_top_k]
    # ... (cross-encoder)
```

---

### 2.4. Фильтрация и постобработка результатов

#### Фильтрация по метаданным

Позволяет искать только в определённых источниках или по дате.

```python
def search(self, query, filter_metadata=None):
    # ...
    if filter_metadata:
        filtered = []
        for doc_idx, score in sorted_items:
            meta = self.metadatas[doc_idx]
            if all(meta.get(k) == v for k, v in filter_metadata.items()):
                filtered.append((doc_idx, score))
        sorted_items = filtered
```

#### Дедупликация по тексту

Удаляет дубликаты, если один и тот же текст попал в несколько чанков.

```python
seen = set()
unique = []
for doc_idx, score in sorted_items:
    text = self.texts[doc_idx]
    if text not in seen:
        seen.add(text)
        unique.append((doc_idx, score))
```

#### Порог отсечения по score

В коде пока нет, но можно добавить:

```python
# Отбрасываем документы с низким скором (например, < 0.3)
threshold = 0.3
filtered_items = [(idx, score) for idx, score in sorted_items if score >= threshold]
```

---

### 2.5. Пример: гибридный поиск находит точное совпадение

**Запрос:** *"Что говорит статья 284.5 НК РФ?"*

- **Чистый векторный поиск** – может вернуть документы о налоге на прибыль, но не обязательно с упоминанием конкретной статьи.
- **Гибридный поиск** – BM25 находит точное совпадение "статья 284.5" в документе, и этот документ поднимается наверх.

**Результат из нашего эксперимента:**

```
Вопрос: Что говорит статья 284.5 НК РФ?
  Векторный: Recall@5=1.000, время=0.011с
  Гибрид+реранкинг: Recall@5=1.000, время=0.348с
```

---

### 2.6. Эксперимент: сравнение методов

**Условия:** 4 запроса из корпуса (смесь фактологических и с кодами). Для каждого запроса измеряем Recall@5 и время.

**Результаты из нашего кода:**

```
============================================================
Средние результаты:
  Векторный: Recall@5=1.000 ± 0.000, время=0.013с
  Гибрид+реранкинг: Recall@5=1.000 ± 0.000, время=0.362с
============================================================
```

**Вывод:** на маленьком датасете (6 документов) оба метода дают Recall@5 = 1.0. Гибридный поиск с реранкингом медленнее (~0.36 с против ~0.01 с), но на больших датасетах он даёт прирост качества.

---

### 2.7. Полный код

```python
# ================================================================
# Тема 2. Расширенный поиск и переранжирование
# Полный код для Google Colab (локальный запуск)
# ================================================================

!pip install -q requests sentence-transformers chromadb rank-bm25

import os
import sys
import time
import json
import requests
import subprocess
import numpy as np
from typing import List, Dict, Tuple, Optional
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
print(f"Среда: {'Colab' if IS_COLAB else 'локально'}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DB_PATH = "/content/drive/MyDrive/chroma_db"
else:
    DB_PATH = "./chroma_db"

MODEL_NAME = "qwen2.5:1.5b"

def ensure_model(model_name):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            models = [m['name'] for m in r.json().get('models', [])]
            if model_name in models:
                print(f"Модель {model_name} уже загружена")
                return True
            else:
                print(f"Загрузка {model_name}...")
                subprocess.run(["ollama", "pull", model_name], check=True, capture_output=True)
                print(f"Модель {model_name} загружена")
                return True
    except Exception as e:
        print(f"Ошибка: {e}")
        return False

def setup_ollama():
    if not IS_COLAB:
        try:
            subprocess.run(["ollama", "--version"], check=True, capture_output=True)
            return ensure_model(MODEL_NAME)
        except:
            print("Ollama не найден, установите вручную")
            return False
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama уже запущен")
        return ensure_model(MODEL_NAME)
    except:
        pass

    print("Установка Ollama...")
    !apt-get install -y zstd pciutils > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh
    os.environ["PATH"] += os.pathsep + "/usr/local/bin"
    try:
        subprocess.run(["ollama", "--version"], check=True, capture_output=True)
    except:
        print("Ошибка установки")
        return False

    print("Запуск сервера...")
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(30):
        try:
            requests.get("http://localhost:11434/api/tags", timeout=2)
            print("Ollama готов")
            break
        except:
            time.sleep(1)
    else:
        print("Сервер не запустился")
        return False
    return ensure_model(MODEL_NAME)

if not setup_ollama():
    sys.exit(1)

def query_ollama(prompt, model=MODEL_NAME, temperature=0.7, max_tokens=512, system=None, retries=3, timeout=300):
    url = "http://localhost:11434/api/generate"
    payload = {"model": model, "prompt": prompt, "stream": False, "options": {"temperature": temperature, "num_predict": max_tokens}}
    if system:
        payload["system"] = system
    for attempt in range(retries):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json().get("response", "")
        except Exception as e:
            print(f"Попытка {attempt+1} ошибка: {e}")
            time.sleep(2 ** attempt)
    return "[Ошибка]"

class OllamaAdapter:
    def __init__(self, model=MODEL_NAME):
        self.model = model
    def generate(self, prompt, system=None, temperature=0.7, max_tokens=512):
        return query_ollama(prompt, self.model, temperature, max_tokens, system)

class AdvancedRetriever:
    def __init__(self, collection, embed_model, texts, metadatas, alpha=0.6, k1=1.2, b=0.75,
                 rerank_top_k=20, final_top_k=5):
        self.collection = collection
        self.embed_model = embed_model
        self.texts = texts
        self.metadatas = metadatas
        self.alpha = alpha
        self.rerank_top_k = rerank_top_k
        self.final_top_k = final_top_k
        tokenized = [doc.split() for doc in texts]
        self.bm25 = BM25Okapi(tokenized, k1=k1, b=b)
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        print("Cross-encoder загружен")

    def _vector_search(self, query, top_k):
        q_emb = self.embed_model.encode([query], normalize_embeddings=True).tolist()
        res = self.collection.query(query_embeddings=q_emb, n_results=top_k)
        ids = res['ids'][0]
        dist = res['distances'][0]
        sim = [1 - d for d in dist]
        return ids, sim

    def _bm25_search(self, query, top_k):
        tok = query.split()
        scores = self.bm25.get_scores(tok)
        indices = np.argsort(scores)[::-1][:top_k]
        return indices, scores

    def search(self, query, filter_metadata=None):
        vec_ids, vec_scores = self._vector_search(query, self.rerank_top_k * 2)
        bm25_indices, all_scores = self._bm25_search(query, self.rerank_top_k * 2)
        max_bm25 = max(all_scores) if all_scores.any() else 1.0

        combined = {}
        for idx, score in zip(vec_ids, vec_scores):
            doc_idx = int(idx.split('_')[1])
            combined[doc_idx] = self.alpha * score
        for doc_idx in bm25_indices:
            score = all_scores[doc_idx] / max_bm25 if max_bm25 > 0 else 0
            combined[doc_idx] = combined.get(doc_idx, 0) + (1 - self.alpha) * score

        sorted_items = sorted(combined.items(), key=lambda x: x[1], reverse=True)

        # дедупликация по тексту
        seen = set()
        unique = []
        for doc_idx, score in sorted_items:
            text = self.texts[doc_idx]
            if text not in seen:
                seen.add(text)
                unique.append((doc_idx, score))
        sorted_items = unique

        if filter_metadata:
            filtered = []
            for doc_idx, score in sorted_items:
                meta = self.metadatas[doc_idx]
                if all(meta.get(k) == v for k, v in filter_metadata.items()):
                    filtered.append((doc_idx, score))
            sorted_items = filtered

        candidates = sorted_items[:self.rerank_top_k]
        if not candidates:
            return []

        candidate_texts = [self.texts[idx] for idx, _ in candidates]
        pairs = [[query, doc] for doc in candidate_texts]
        rerank_scores = self.reranker.predict(pairs)

        final_indices = np.argsort(rerank_scores)[::-1][:self.final_top_k]
        results = []
        for i in final_indices:
            doc_idx = candidates[i][0]
            results.append({
                'doc_idx': doc_idx,
                'text': self.texts[doc_idx],
                'metadata': self.metadatas[doc_idx],
                'score': float(rerank_scores[i])
            })
        return results

class RAGSystemAdvanced:
    def __init__(self, llm_adapter, embed_model, texts, metadatas, db_path=DB_PATH):
        self.llm = llm_adapter
        self.embed_model = embed_model
        self.texts = texts
        self.metadatas = metadatas
        self.client = chromadb.PersistentClient(path=db_path)
        self.collection = None
        self._init_collection()
        self.retriever = AdvancedRetriever(
            self.collection, self.embed_model, texts, metadatas,
            alpha=0.6, rerank_top_k=20, final_top_k=5
        )

    def _init_collection(self):
        try:
            self.collection = self.client.get_collection("documents")
            print(f"Загружено документов: {self.collection.count()}")
        except:
            self.collection = self.client.create_collection("documents")
            print("Создана новая коллекция")

    def add_documents(self, documents, metadatas=None):
        if metadatas is None:
            metadatas = [{} for _ in documents]
        embeddings = self.embed_model.encode(documents, normalize_embeddings=True).tolist()
        ids = [f"doc_{i}" for i in range(len(documents))]
        self.collection.add(documents=documents, embeddings=embeddings, metadatas=metadatas, ids=ids)
        print(f"Добавлено {len(documents)} документов")

    def query(self, question, filter_metadata=None, system_prompt=None):
        retrieved = self.retriever.search(question, filter_metadata=filter_metadata)
        if not retrieved:
            return "Нет релевантных документов."
        context = ""
        for i, doc in enumerate(retrieved):
            src = doc['metadata'].get('source', 'неизвестно')
            context += f"\n[Документ {i+1}] Источник: {src}\n{doc['text']}\n"
        default_system = "Ты — юридический консультант. Отвечай строго по контексту. Если ответа нет в контексте, скажи об этом."
        system = system_prompt or default_system
        prompt = f"Контекст:\n{context}\n\nВопрос: {question}\nОтвет:"
        return self.llm.generate(prompt, system=system, temperature=0.3, max_tokens=512)

    def search_only(self, question, filter_metadata=None):
        return self.retriever.search(question, filter_metadata=filter_metadata)

def compute_recall_at_k(retrieved_ids, relevant_ids, k=5):
    if not relevant_ids:
        return 0.0
    retrieved_set = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)
    return len(retrieved_set & relevant_set) / len(relevant_set)

def demo():
    print("="*60)
    print("Демонстрация гибридного поиска и реранкинга с метриками")
    print("="*60)

    docs = [
        "Самозанятые граждане платят налог на профессиональный доход (НПД). Ставка 4% при работе с физлицами и 6% с юрлицами.",
        "Для российских IT-компаний с аккредитацией Минцифры налог на прибыль 0% до конца 2024, с 2025 — 5%.",
        "Страховые взносы для самозанятых необязательны, можно платить добровольно для пенсии.",
        "IT-компании освобождены от НДС при продаже собственного ПО.",
        "Налоговый кодекс РФ, статья 284.5 устанавливает пониженные ставки налога на прибыль для IT-компаний.",
        "ФЗ-422 о налоге на профессиональный доход регулирует ставки для самозанятых."
    ]
    metas = [
        {"source": "Закон о НПД (ФЗ-422)"},
        {"source": "НК РФ (Ст. 284.5)"},
        {"source": "Закон о НПД (Ст. 14)"},
        {"source": "НК РФ (Ст. 145.1)"},
        {"source": "НК РФ (Ст. 284.5)"},
        {"source": "ФЗ-422"}
    ]

    embed_model = SentenceTransformer("cointegrated/rubert-tiny2", device="cpu")
    llm = OllamaAdapter(MODEL_NAME)
    rag = RAGSystemAdvanced(llm, embed_model, docs, metas)

    # очистка и заполнение БД
    try:
        if rag.collection.count() > 0:
            ids = rag.collection.get()['ids']
            if ids:
                rag.collection.delete(ids)
    except:
        pass
    rag.add_documents(docs, metas)

    ground_truth = {
        "Какой налог платят самозанятые?": [0, 5],
        "Ставка налога на прибыль для IT-компаний?": [1, 4],
        "Что говорит статья 284.5 НК РФ?": [4],
        "Обязательны ли страховые взносы для самозанятых?": [2]
    }

    questions = list(ground_truth.keys())
    vec_recalls, hybrid_recalls, vec_times, hybrid_times = [], [], [], []

    for q in questions:
        print(f"\nВопрос: {q}")
        relevant = ground_truth[q]

        # векторный поиск
        start = time.time()
        q_emb = embed_model.encode([q], normalize_embeddings=True).tolist()
        vec_res = rag.collection.query(query_embeddings=q_emb, n_results=5)
        vec_time = time.time() - start
        vec_ids = [int(id.split('_')[1]) for id in vec_res['ids'][0]]
        recall_vec = compute_recall_at_k(vec_ids, relevant, k=5)
        vec_recalls.append(recall_vec)
        vec_times.append(vec_time)

        # гибридный + реранкинг
        start = time.time()
        hybrid_res = rag.search_only(q)
        hybrid_time = time.time() - start
        hybrid_ids = [doc['doc_idx'] for doc in hybrid_res]
        recall_hybrid = compute_recall_at_k(hybrid_ids, relevant, k=5)
        hybrid_recalls.append(recall_hybrid)
        hybrid_times.append(hybrid_time)

        print(f"  Векторный: Recall@5={recall_vec:.3f}, время={vec_time:.3f}с")
        print(f"  Гибрид+реранкинг: Recall@5={recall_hybrid:.3f}, время={hybrid_time:.3f}с")

    print("\n" + "="*60)
    print("Средние результаты:")
    print(f"  Векторный: Recall@5={np.mean(vec_recalls):.3f} ± {np.std(vec_recalls):.3f}, время={np.mean(vec_times):.3f}с")
    print(f"  Гибрид+реранкинг: Recall@5={np.mean(hybrid_recalls):.3f} ± {np.std(hybrid_recalls):.3f}, время={np.mean(hybrid_times):.3f}с")
    print("="*60)

if __name__ == "__main__":
    demo()
```

**Результат выполнения:**

```
Среда: Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ollama уже запущен
Модель qwen2.5:1.5b уже загружена
============================================================
Демонстрация гибридного поиска и реранкинга с метриками
============================================================
Loading weights: 100%
55/55 [00:00<00:00, 1700.00it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED |
cls.predictions.transform.dense.weight     | UNEXPECTED |
...
Notes:
- UNEXPECTED: can be ignored when loading from different task/architecture
Загружено документов: 6
Loading weights: 100%
105/105 [00:00<00:00, 2080.52it/s]
Cross-encoder загружен
Добавлено 6 документов

Вопрос: Какой налог платят самозанятые?
  Векторный: Recall@5=1.000, время=0.017с
  Гибрид+реранкинг: Recall@5=1.000, время=0.361с

Вопрос: Ставка налога на прибыль для IT-компаний?
  Векторный: Recall@5=1.000, время=0.011с
  Гибрид+реранкинг: Recall@5=1.000, время=0.356с

Вопрос: Что говорит статья 284.5 НК РФ?
  Векторный: Recall@5=1.000, время=0.011с
  Гибрид+реранкинг: Recall@5=1.000, время=0.348с

Вопрос: Обязательны ли страховые взносы для самозанятых?
  Векторный: Recall@5=1.000, время=0.011с
  Гибрид+реранкинг: Recall@5=1.000, время=0.381с

============================================================
Средние результаты:
  Векторный: Recall@5=1.000 ± 0.000, время=0.013с
  Гибрид+реранкинг: Recall@5=1.000 ± 0.000, время=0.362с
============================================================
```

---

### 2.8. Контрольные вопросы

1. *Почему гибридный поиск (BM25 + эмбеддинги) лучше чистого векторного для запросов с кодами?*  
   **Ответ:** BM25 учитывает точное вхождение терминов и их частоту, поэтому запросы с кодами и номерами получают высокую оценку, даже если семантический эмбеддинг не различает их.

2. *В чём отличие cross‑encoder от bi‑encoder (Sentence‑BERT) и почему реранкинг даёт улучшение?*  
   **Ответ:** Cross‑encoder обрабатывает пару (запрос, документ) совместно, что позволяет учитывать взаимодействие между ними, но медленнее. Bi‑encoder кодирует их отдельно и быстрее. Реранкинг использует cross‑encoder на небольшом множестве кандидатов для более точного ранжирования.

3. *Какой компромисс нужно учитывать при использовании реранкинга в продакшене?*  
   **Ответ:** Реранкинг увеличивает время ответа (так как требует дополнительных вычислений cross‑encoder). Поэтому его применяют только к топ‑N кандидатам (например, 20), и нужно балансировать точность и задержку.

---

### 2.9. Задания

1. **Реализуйте гибридный поиск** в вашем существующем RAG-пайплайне и сравните качество (Recall@5, MRR) с чистым векторным на 10 запросах. Постройте таблицу.

2. **Добавьте реранкинг** с cross‑encoder и оцените улучшение относительно гибридного поиска. Замерьте время выполнения обоих методов. Напишите выводы.

---

### 2.10. Список литературы

- **Robertson, S., Zaragoza, H.** (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. – Foundations and Trends in Information Retrieval.
- **Cross-Encoder models** – Hugging Face documentation: https://huggingface.co/cross-encoder
- **Sentence-Transformers**: https://www.sbert.net/
- **rank_bm25** library: https://github.com/dorianbrown/rank_bm25

---

В этом разделе мы рассмотрели методы улучшения поиска: гибридный поиск (векторный + BM25), переранжирование с cross‑encoder, многопроходный поиск, фильтрацию и дедупликацию. Комбинация этих техник позволяет значительно повысить качество извлечения релевантных документов, что в итоге улучшает ответы всей RAG-системы. В следующей теме мы перейдём к оптимизации генерации и работе с длинными контекстами.